## Init session

In [ ]:
%load_ext autoreload
%autoreload 2

### Install dependencies

In [ ]:
!chmod +x install.sh
! ./install.sh > /dev/null 2>&1

### Import packages

In [ ]:
import os
import boto3
import subprocess

from pathlib import Path
from random import randint

from rich.pretty import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import torch


from sklearn.model_selection import train_test_split

import model as ic
from datetime import datetime, timedelta

## Download from S3 bucket

In [ ]:
# 1. Automatically retrieve credentials from the Onyxia environment
mybucket = "jbarrere"
key = os.environ.get('AWS_ACCESS_KEY_ID')
secret = os.environ.get('AWS_SECRET_ACCESS_KEY')
token = os.environ.get('AWS_SESSION_TOKEN')
endpoint = os.environ.get('AWS_ENDPOINT_URL')

# 2. Initialize the client
s3 = boto3.client("s3",endpoint_url = endpoint,
                  aws_access_key_id = key, 
                  aws_secret_access_key = secret, 
                  aws_session_token = token)

# 3. Check connection
listobjbucket = s3.list_objects_v2(Bucket=mybucket, MaxKeys=5)
if 'Contents' in listobjbucket:
    print("Connection OK!\nFiles:")
    for obj in listobjbucket['Contents']:
        print("-", obj['Key'])

In [ ]:
# Download data.zip if needed
if not os.path.exists("/home/onyxia/work/data.zip"):
    s3.download_file(mybucket, "data.zip", "data.zip")

# Unzip data.zip if needed
if not os.path.exists("/home/onyxia/work/data"):
    subprocess.run(["unzip", "data.zip"],
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL)
    print("Done!")

# Download mlflow if needed
if not os.path.exists("/home/onyxia/work/mlflow.db"):
    s3.download_file(mybucket, "mlflow.db", "mlflow.db")

# Download mlruns if needed
if not os.path.exists("/home/onyxia/work/mlruns.zip"):
    s3.download_file(mybucket, "mlruns.zip", "mlruns.zip")

# Unzip data.zip if needed
if not os.path.exists("/home/onyxia/work/mlruns"):
    subprocess.run(["unzip", "mlruns.zip"],
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL)
    print("Done!")



## Load & clean data

### Import annotations

In [ ]:
binary_columns_todrop = ["animal", "no", "water", "zoom"]
binary_columns = ["human", "anthropic", "vegetation", "rock", "snow"]
tab_raw = pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv")).drop(
    binary_columns_todrop, axis = 1
)
print(tab_raw)

### Select sites

In [ ]:
sites_trainvaltest = ["Carpathians", "French_Alps", "Stubai_Valley", "Vinschgau"]
sites_external = ["Danube", "Dovre", "Sierra_Nevada"]

tab_external = tab_raw[tab_raw["site"].isin(sites_external)].copy()
tab_raw = tab_raw[tab_raw["site"].isin(sites_trainvaltest)]

### Rebalance categories

In [ ]:
n = 2000
rng = np.random.default_rng(42)           # ------------------------------------------------------------------ #
# Compute weights from imbalance in the original dataframe
# ------------------------------------------------------------------ #
# p = proportion of 1s; distance from 0.5 ranges from 0 (perfect balance)
# to 0.5 (all 0s or all 1s). We map it to a weight >= 1.
# weight = 1 + k * (|p - 0.5| / 0.5)  with k controlling the max weight.
k = 4  # max additional weight on top of the baseline 1
weights = {}
for col in binary_columns:
    p = tab_raw[col].mean()
    imbalance = abs(p - 0.5) / 0.5  # 0 = perfectly balanced, 1 = fully skewed
    weights[col] = 1 + k * imbalance
    
# ------------------------------------------------------------------ #
# Greedy balanced selection
# ------------------------------------------------------------------ #
seed = 42  
rng = np.random.default_rng(seed)
df_shuffled = tab_raw.sample(frac=1, random_state=seed).reset_index(drop=True)

selected_indices = []
counts = {col: {0: 0, 1: 0} for col in binary_columns}
target = n // 2

base_tolerance = 500   
ramp = 1000

values = df_shuffled[binary_columns].values

for idx, row_vals in enumerate(values):
    if len(selected_indices) >= n:
        break

    score = 0

    for j, col in enumerate(binary_columns):
        val = int(row_vals[j])
        w = weights[col]

        score += w * (
            (target - counts[col][val])
            - (target - counts[col][1 - val])
        )

    # -------- trade-off control --------
    progress = len(selected_indices) / n
    threshold = -(base_tolerance + progress * ramp)

    if score >= threshold:
        selected_indices.append(idx)

        for j, col in enumerate(binary_columns):
            counts[col][int(row_vals[j])] += 1

In [ ]:
# Make two entry datasets : one balanced and one with all data
tab_balanced = df_shuffled.loc[selected_indices].reset_index(drop = True)
tab_full = df_shuffled.copy()

In [ ]:
# Show how categories are balanced (or not)
# - Balanced dataframe
pd.DataFrame(
    data={
        "Category": [col for col in binary_columns],
        "%": [tab_balanced[col].mean() * 100 for col in binary_columns],
        "Number": [sum(tab_balanced[col]) for col in binary_columns],
        "Total": len(tab_balanced),
        "Data": "balanced"
    }
).sort_values("%", ascending = False)

In [ ]:
# Full dataframe
pd.DataFrame(
    data={
        "Category": [col for col in binary_columns],
        "%": [tab_full[col].mean() * 100 for col in binary_columns],
        "Number": [sum(tab_full[col]) for col in binary_columns],
        "Total": len(tab_full),
        "Data": "full"
    }
).sort_values("%", ascending = False)

## Minigrid experiment

In [ ]:
def run_minigrid(tab, exp_name):
    """
    Run the complete experiment pipeline with stratification and hyperparameter tuning.
    
    Parameters:
    -----------
    tab : pd.DataFrame
        The input dataframe containing the data
    exp_name : str
        The name of the experiment (used for logging)
    """
    # Stratify dataset
    tab_strat = tab.copy()
    tab_strat["strat"] = ""
    for col in tab.columns[2:]:
        tab_strat["strat"] += tab_strat[col].astype(str)

    # Split in train and validation dataset
    trainval, test = train_test_split(tab_strat, test_size=0.15, random_state=42, stratify=tab_strat["strat"])
    test = test.drop("strat", axis=1)

    # Set the backbones, repetitions, learning rates and batch size to experiment
    backbones = ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]
    repetitions = [3, 4, 5] 
    learning_rates = [0.0001, 0.00001]
    batch_sizes = [16, 32]

    # Loop on all parameters
    for rep in repetitions:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
                
        for backbone in backbones:
            for lr in learning_rates:
                for bs in batch_sizes:
                        
                    ic.train_model(
                        train_data=train,
                        val_data=val,
                        batch_size=bs,
                        max_epochs=20,  
                        image_size=224,
                        run_owner="onyxia",
                        exp_name=exp_name,
                        backbone=backbone,
                        loss_name="bce",
                        loss_params={"alpha":0.25, "gamma":2},  
                        device=ic.get_device(),
                        checkpoints_n_saved=1,
                        learning_rate=lr,
                        early_stoper_patience=5,
                        early_stoper_min_delta=0.001,
                        use_lr_finder=False,
                        lr_scheduler_step=10,
                        lr_scheduler_gamma=0.85,
                        print_steps="print",
                        log_progress=False,
                        plot_loss=False,
                        num_workers=10,
                    )

In [ ]:
# Run minigrid experiment with balanced dataset
run_minigrid(tab=tab_balanced, exp_name="minigrid_balanced")

In [ ]:
# Run minigrid experiment with full dataset
run_minigrid(tab=tab_full, exp_name="minigrid_full")

### Export minigrid outputs

In [ ]:
def extract_mlflow_results_minigrid(experiment_names, output_file):
    """
    Extract and combine MLflow run results from specified experiments.
    
    Parameters:
    -----------
    experiment_names : list or str
        Name(s) of the MLflow experiment(s) to search
    output_file : str
        Path where the combined results CSV should be saved
    
    Returns:
    --------
    pd.DataFrame
        The combined results dataframe
    """
    # Convert single experiment name to list if needed
    if isinstance(experiment_names, str):
        experiment_names = [experiment_names]
    
    # Search for runs
    runs = mlflow.search_runs(search_all_experiments=True, experiment_names=experiment_names)

    ### - Dirty fix since backbone was not logged (to remove afterwards)
    runs_df = runs[[ "run_id", "start_time", "params.batch_size", "params.learning_rate" ]].sort_values("start_time").reset_index(drop=True).copy()
    k=0
    for rep in [3, 4, 5]:
        for backbone in ["hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16"]:
            for lr in [0.0001, 0.00001]:
                for bs in [16, 32]:
                    runs_df.loc[k, 'backbone'] = backbone
                    k=k+1
    runs_df = runs_df.drop(columns="start_time")
    runs_df.columns = ["run_id", "batch_size", "learning_rate", "backbone"]
    ### - End of the dirty fix
    
    # Extract relevant parameters
    #runs_df = runs[[
    #    "run_id",
    #    "params.backbone",
    #    "params.batch_size",
    #    "params.learning_rate"
    #]].copy()
    #runs_df.columns = ["run_id", "backbone", "batch_size", "learning_rate"]
    
    # Collect all classification reports
    all_data = []
    for _, row in runs_df.iterrows():
        run_id = row["run_id"]
        
        try:
            local_path = mlflow.artifacts.download_artifacts(
                run_id=run_id,
                artifact_path="metrics/classification_report.csv"
            )
            
            df_report = pd.read_csv(local_path, sep=";")
            
            df_report["run_id"] = run_id
            df_report["backbone"] = row["backbone"]
            df_report["batch_size"] = row["batch_size"]
            df_report["learning_rate"] = row["learning_rate"]
            
            all_data.append(df_report)
            
        except Exception as e:
            print(f"Error for {run_id}: {e}")
    
    # Combine all results
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Reorder columns
    cols_order = ["run_id", "backbone", "batch_size", "learning_rate"]
    final_df = final_df[
        cols_order + [col for col in final_df.columns if col not in cols_order]
    ]
    
    # Save to CSV
    final_df.to_csv(output_file, index=False)
    
    return final_df





In [ ]:
# Extract output for the balanced minigrid
os.makedirs("outputs", exist_ok=True)
df_minigrid_balanced = extract_mlflow_results_minigrid(
    experiment_names=["minigrid_balanced"],
    output_file="outputs/results_minigrid_balanced.csv"
)
print(df_minigrid_balanced)

In [ ]:
# Extract output for the full minigrid
df_minigrid_full = extract_mlflow_results_minigrid(
    experiment_names=["minigrid_full"],
    output_file="outputs/results_minigrid_full.csv"
)
print(df_minigrid_full)

### Train 50 models for hf_swt_t

In [ ]:
def train_models(tab, backbone, learning_rate, batch_size, exp_name):
    """
    Train a model with specified hyperparameters.
    
    Parameters:
    -----------
    tab : pd.DataFrame
        The input dataframe containing the data
    backbone : str
        The backbone model to use (e.g., "hf_swt_t", "hf_resnet", "hf_cnx2_t", "hf_vit_g16")
    learning_rate : float
        The learning rate for training (e.g., 0.0001, 0.00001)
    batch_size : int
        The batch size for training (e.g., 16, 32)
    exp_name : str
        The name of the experiment (used for logging)
    
    """
    # Stratify dataset
    tab_strat = tab.copy()
    tab_strat["strat"] = ""
    for col in tab.columns[2:]:
        tab_strat["strat"] += tab_strat[col].astype(str)

    # Split in train and validation dataset
    trainval, test = train_test_split(tab_strat, test_size=0.15, random_state=42, stratify=tab_strat["strat"])
    test = test.drop("strat", axis=1)

    # Loop over different random seeds for train/val splits
    for rep in [11, 12, 13, 14, 15]:
        train, val = train_test_split(
            trainval,
            test_size=0.18,
            stratify=trainval["strat"],
            random_state=rep
        )
        train = train.drop("strat", axis=1)
        val = val.drop("strat", axis=1)
        
        # Train multiple times with same split (10 repetitions)
        for _ in list(range(10)):
            ic.train_model(
                train_data=train,
                val_data=val,
                batch_size=batch_size,  
                max_epochs=100,
                image_size=224,
                run_owner="onyxia",
                exp_name=exp_name+str(rep),  
                backbone=backbone,  
                loss_name="bce",
                loss_params={"alpha": 0.25, "gamma": 2},
                device=ic.get_device(),
                checkpoints_n_saved=1,
                learning_rate=learning_rate,  
                early_stoper_patience=10,
                early_stoper_min_delta=0.001,
                use_lr_finder=False,
                lr_scheduler_step=10,
                lr_scheduler_gamma=0.85,
                print_steps="print",
                log_progress=False,
                plot_loss=False,
                num_workers=10,
            )

In [ ]:
# Train model with balanced dataset
train_models(
    tab=tab_balanced,
    backbone="hf_swt_t",
    learning_rate=0.0001,
    batch_size=16,
    exp_name="swin_balanced"
)

In [ ]:
# Train model with full dataset
train_models(
    tab=tab_full,
    backbone="hf_swt_t",
    learning_rate=0.0001,
    batch_size=16,
    exp_name="swin_full"
)

### Export trainswt outputs

In [ ]:
def export_mlflow_results_swin(exp_name, output_file, repetitions=None):
    """
    Export MLflow results from experiments where repetition is included in the experiment name.
    
    Parameters:
    -----------
    exp_name : str
        The base name of the experiment (e.g., "trainswt", "trainresnet")
    output_file : str
        Path where the combined results CSV should be saved
    repetitions : list, optional
        List of repetition numbers (default: [11, 12, 13, 14, 15])
    
    Returns:
    --------
    pd.DataFrame
        The combined results dataframe
    """
    if repetitions is None:
        repetitions = [11, 12, 13, 14, 15]
    
    all_data = []
    
    for rep in repetitions:
        # Get experiment by name with repetition number appended
        exp_name_full = f"{exp_name}{rep}"
        exp = mlflow.get_experiment_by_name(exp_name_full)
        
        if exp is None:
            print(f"Warning: Experiment '{exp_name_full}' not found. Skipping...")
            continue
        
        runs = mlflow.search_runs(experiment_ids=[exp.experiment_id])
        
        for _, row in runs.iterrows():
            run_id = row["run_id"]
            
            try:
                local_path = mlflow.artifacts.download_artifacts(
                    run_id=run_id,
                    artifact_path="metrics/classification_report.csv"
                )
                
                df_report = pd.read_csv(local_path, sep=";")
                
                df_report["run_id"] = run_id
                df_report["rep"] = rep
                
                all_data.append(df_report)
                
            except Exception as e:
                print(f"Error for {run_id}: {e}")
    
    if not all_data:
        print(f"No data found for experiments with base name: {exp_name}")
        return None
    
    final_df = pd.concat(all_data, ignore_index=True)
    
    cols_order = ["run_id", "rep"]
    final_df = final_df[
        cols_order + [col for col in final_df.columns if col not in cols_order]
    ]
    
    final_df.to_csv(output_file, index=False)
    print(f"Results saved to {output_file}")
    print(f"Total runs exported: {len(final_df)}")
    
    return final_df

In [ ]:
# Export results with balanced dataset
results_swin_balanced = export_mlflow_results_swin(
    exp_name="swin_balanced",
    output_file="outputs/swin_balanced_results.csv",
    repetitions=[11, 12, 13, 14, 15]
)
print(results_swin_balanced)

In [ ]:
# Export results with full dataset
results_swin_full = export_mlflow_results_swin(
    exp_name="swin_full",
    output_file="outputs/swin_full_results.csv",
    repetitions=[11, 12, 13, 14, 15]
)
print(results_swin_full)

### Find best model

In [ ]:
# Function to get the id of the best model
def get_best_model_id(final_df, exp_name, days=3): 

    # Subset the df with weighted average only
    df_weighted = final_df[final_df["labels"] == "weighted avg"].copy()

    # Search for experiments
    experiments = mlflow.search_experiments()
    experiment_ids = [e.experiment_id for e in experiments if e.name.startswith(exp_name)]
    
    # Calculate the cutoff time (3 days ago from now)
    cutoff_time = datetime.now() - timedelta(days=days)
    cutoff_time_ms = int(cutoff_time.timestamp() * 1000)  # MLflow uses milliseconds
    
    # Search for runs, filtering by start time
    runs = mlflow.search_runs(
        experiment_ids=experiment_ids,
        filter_string=f"attributes.start_time >= {cutoff_time_ms}"
    )

    # If no runs found in the last 3 days, handle gracefully
    if runs.empty:
        print(f"No runs found in the last {days} days for experiment {exp_name}")
        return None

    # Get the ID of the best model
    best = runs.sort_values("params.F1_weighted_avg", ascending=False).iloc[0]
    best_run_id = best.run_id

    # Return best id
    return best_run_id

In [ ]:
# Get best model ID for swin full and swin_balanced
best_run_id_balanced = get_best_model_id(final_df = results_swin_balanced, exp_name = "swin_balanced")
best_run_id_full = get_best_model_id(final_df = results_swin_full, exp_name = "swin_full")
print(best_run_id_balanced)
print(best_run_id_full)

### Export training outputs

In [ ]:
def export_metric_history(best_run_id, filename):
    """
    Export metric history for a given MLflow run to a CSV file.
    
    Args:
        best_run_id (str): The MLflow run ID to fetch metrics for
        filename (str): The name/path of the file to export the CSV to
    """
    client = mlflow.tracking.MlflowClient()
    
    train_loss = client.get_metric_history(best_run_id, "training Loss")
    val_loss   = client.get_metric_history(best_run_id, "validation Loss")
    
    train_f1 = client.get_metric_history(best_run_id, "training F1")
    val_f1   = client.get_metric_history(best_run_id, "validation F1")
    
    
    train_loss_df = pd.DataFrame([
        {"epoch": m.step, "train_loss": m.value}
        for m in train_loss
    ])
    
    val_loss_df = pd.DataFrame([
        {"epoch": m.step, "val_loss": m.value}
        for m in val_loss
    ])
    
    train_f1_df = pd.DataFrame([
        {"epoch": m.step, "train_f1": m.value}
        for m in train_f1
    ])
    
    val_f1_df = pd.DataFrame([
        {"epoch": m.step, "val_f1": m.value}
        for m in val_f1
    ])
    
    df_epoch = (
        train_loss_df
        .merge(val_loss_df, on="epoch", how="outer")
        .merge(train_f1_df, on="epoch", how="outer")
        .merge(val_f1_df, on="epoch", how="outer")
        .sort_values("epoch")
    )
    
    df_epoch.to_csv(filename, index=False)
    
    return df_epoch


In [ ]:
# Get metric history for full and balanced training
metric_history_balanced = export_metric_history(
    best_run_id_balanced, "outputs/best_model_curves_balanced.csv")
metric_history_full = export_metric_history(
    best_run_id_full, "outputs/best_model_curves_full.csv")

## Validation

### Export best model predictions

In [ ]:
# Function to export tresholds, and the predictons of train, val, test and external
def export_outputs_bestmodel(tab, tab_external, best_run_id, name): 

    # Stratify dataset
    tab_strat = tab.copy()
    tab_strat["strat"] = ""
    for col in tab.columns[2:]:
        tab_strat["strat"] += tab_strat[col].astype(str)

    # Split in train and validation dataset
    trainval, test = train_test_split(tab_strat, test_size=0.15, random_state=42, stratify=tab_strat["strat"])
    train, val = train_test_split(trainval,test_size=0.18,stratify=trainval["strat"],random_state=11)
    test = test.drop("strat", axis=1)

    # Load best model
    model = mlflow.pytorch.load_model(
        f"runs:/{best_run_id}/model",
        map_location=torch.device(ic.get_device()),
    )

    # Export tresholds
    pd.DataFrame(model.thresholds).to_csv(f"outputs/thresholds_{name}.csv", index=False)

    # Export train predictions
    os.makedirs("outputs/train", exist_ok=True)
    train = train.drop("strat", axis=1)
    proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=train, train_mode=False)))
    proba.to_csv(f"outputs/train/train_proba_{name}.csv", index=False)
    trainout = model.get_val_data(dataset=ic.FldDataset(data=train, train_mode=False))
    trainout["predictions_revue"].to_csv(f"outputs/train/train_prediction_revue_{name}.csv", index=False)
    trainout["classification_report"].to_csv(f"outputs/train/train_classification_report_{name}.csv", index=False)

    # Export val predictions
    os.makedirs("outputs/val", exist_ok=True)
    val = val.drop("strat", axis=1)
    proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=val, train_mode=False)))
    proba.to_csv(f"outputs/val/val_proba_{name}.csv", index=False)
    valout = model.get_val_data(dataset=ic.FldDataset(data=val, train_mode=False))
    valout["predictions_revue"].to_csv(f"outputs/val/val_prediction_revue_{name}.csv", index=False)
    valout["classification_report"].to_csv(f"outputs/val/val_classification_report_{name}.csv", index=False)

    # Export test predictions
    os.makedirs("outputs/test", exist_ok=True)
    proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=test, train_mode=False)))
    proba.to_csv(f"outputs/test/test_proba_{name}.csv", index=False)
    testout = model.get_val_data(dataset=ic.FldDataset(data=test, train_mode=False))
    testout["predictions_revue"].to_csv(f"outputs/test/test_prediction_revue_{name}.csv", index=False)
    testout["classification_report"].to_csv(f"outputs/test/test_classification_report_{name}.csv", index=False)

    # Export external predictions
    os.makedirs("outputs/external", exist_ok=True)
    proba =  pd.DataFrame(model.predict_propabilities(dataset=ic.FldDataset(data=tab_external, train_mode=False)))
    proba.to_csv(f"outputs/external/external_proba_{name}.csv", index=False)
    externalout = model.get_val_data(dataset=ic.FldDataset(data=tab_external, train_mode=False))
    externalout["predictions_revue"].to_csv(f"outputs/external/external_prediction_revue_{name}.csv", index=False)
    externalout["classification_report"].to_csv(f"outputs/external/external_classification_report_{name}.csv", index=False)


In [ ]:
# Exports for balanced model
export_outputs_bestmodel(tab = tab_balanced, tab_external = tab_external, 
                         best_run_id = best_run_id_balanced, name = "balanced")
# Exports for full model
export_outputs_bestmodel(tab = tab_full, tab_external = tab_external, 
                         best_run_id = best_run_id_full, name = "full")

### Export external predictions per site

In [ ]:
def export_predictions_persite(tab, sites, best_run_id, name): 

    """
    Export the predictions of a given model (best_run_id) for each site separately
    
    Args:
        tab (pd.DataFrame): The input dataframe containing the data
        best_run_id (str): The MLflow run ID to fetch metrics for
        sites (list): Name of all the sites to predict
        name (str): name to include at the end of the file saved ("full" or "balanced")
        
    """
    
    # Load best model
    model = mlflow.pytorch.load_model(
        f"runs:/{best_run_id}/model",
        map_location=torch.device(ic.get_device()),
    )

    # Loop on all sites
    for site in sites:

        # Create directory for each site if needed
        out_dir = f"outputs/sites/{site}"
        os.makedirs(out_dir, exist_ok=True)

        # Subset the dataset for the site
        subset = tab[tab["site"] == site].reset_index(drop=True)
    
        # Convert into FldDataset
        dataset = ic.FldDataset(data=subset, train_mode=False)

        # probabilities
        proba = pd.DataFrame(model.predict_propabilities(dataset=dataset))
        proba.to_csv(f"{out_dir}/proba_{name}.csv", index=False)

        # model outputs
        challengeout = model.get_val_data(dataset=dataset)
        challengeout["predictions_revue"].to_csv(
            f"{out_dir}/prediction_revue_{name}.csv",
            index=False
        )
        challengeout["classification_report"].to_csv(
            f"{out_dir}/classification_report_{name}.csv",
            index=False
        )

In [ ]:
export_predictions_persite(tab = tab_external, sites = sites_external, best_run_id = best_run_id_balanced, name = "balanced")
export_predictions_persite(tab = tab_external, sites = sites_external, best_run_id = best_run_id_full, name = "full")

### Prepare files for export

In [ ]:
# Remove original mlruns.zip if exists
if os.path.exists("mlruns.zip"): 
    subprocess.run(["rm", "mlruns.zip"], check=True)

# Remove data folder to make room in the directory
if os.path.exists("data"): 
    subprocess.run(["rm", "-rf", "data"], check=True)
    
# Zip mlruns and outputs
subprocess.run(["zip", "-rq", "outputs.zip", "outputs"], check=True)
subprocess.run(["zip", "-rq", "mlruns.zip", "mlruns"], check=True)

## Inference

In [ ]:
# Download allsites.zip if needed
if not os.path.exists("/home/onyxia/work/allsites.zip"):
    s3.download_file(mybucket, "allsites.zip", "allsites.zip")

# Download mlflow if needed
if not os.path.exists("/home/onyxia/work/mlflow.db"):
    s3.download_file(mybucket, "mlflow.db", "mlflow.db")

# Download mlruns if needed
if not os.path.exists("/home/onyxia/work/mlruns.zip"):
    s3.download_file(mybucket, "mlruns.zip", "mlruns.zip")



In [ ]:
# Unzip allsites.zip if needed
if not os.path.exists("/home/onyxia/work/allsites"):
    os.makedirs("/home/onyxia/work/allsites", exist_ok=True)
    subprocess.run(["unzip", "allsites.zip", "-d", "/home/onyxia/work/allsites"],
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL)

# Unzip mlruns.zip if needed
if not os.path.exists("/home/onyxia/work/mlruns"):
    subprocess.run(["unzip", "mlruns.zip"],
                   stdout=subprocess.DEVNULL,
                   stderr=subprocess.DEVNULL)

In [ ]:
# Load best model
model = mlflow.pytorch.load_model(
    f"runs:/736f8e3b69c54f9291c78d37e306a86c/model",
    map_location=torch.device(ic.get_device()),
)


In [ ]:
# Prepare dataset for inference
# - Path where the files are stored
dir_path = path = (Path(".").joinpath("allsites"))
# - list all the files in this path 
files = list(dir_path.glob('*.jpg'))
# - Convert into a dataset
dataset_inf = ic.FldDataset(data = files, train_mode=False,test_mode=False)
# - Show the first element of the dataset
dataset_inf[1]

In [ ]:
# Make predictions
# - Predict the labels for each image
pred_inf = model.predict_labels(dataset_inf)
# - Convert into a pd dataframe
df_pred = pd.DataFrame(pred_inf)
# - give column names
df_pred.columns = ["human", "anthropic", "vegetation", "rock", "snow"]
# - add the image id as an additional column
df_pred['images'] = [p.stem for p in files]
# - show dataframe
df_pred

In [ ]:
# Export inference
df_pred.to_csv(f"inference_10sites.csv", index=False)